## Conducting a LiRA attack
In this notebook, we will demonstrate how to use our library to run attacks on a specific set of target examples.

The data used in this notebook is created by following the shadow model training process in the LiRA evaluation setting (training M shadow models which randomly includes half of the training data points, explained more throughly in `examples/mialib_example_2_lira_evaluation.ipynb`).

In [1]:
# Import the library
import numpy as np

from mialibrary.data_loader import load_data_lira
from mialibrary.lira.check_types import check_types
from mialibrary.lira.validate_inputs import validate_inputs
from mialibrary.lira.simple_lira import get_shadow_stats, get_indices_target, run_simple_lira

### Step 1: Load Data

The expected format and shapes are:

- `stat`: Pre-computed statistics from model outputs
    - Format: list of NumPy arrays, NumPy arrays, or JAX arrays
    - Shape of stat: (m_models, n_samples, k_augmentations)

- `in_indices`: Training data indices
    - Format: list of NumPy arrays, NumPy arrays, or JAX arrays
    - Shape of in_indices: (m_models, n_samples)
    - Values: Boolean (True: used in training, False: not used)


In [2]:
# Define paths to your data

stat_path = "data_small/stat_small_patch_camelyon_vit-b-16_-1_16384_-1.0_22_22.pkl"
in_indices_path = "data_small/in_indices_small_patch_camelyon_vit-b-16_-1_16384_-1.0_22_22.pkl"

# Load stat and in_indices data from files.
stat, in_indices = load_data_lira(stat_path, in_indices_path)

# Check the types of the input arrays and convert them to numpy arrays.
stat, in_indices = check_types(stat, in_indices)

# Validate the inputs for shape, NaN values, and number of examples. Convert NumPy arrays to JAX arrays after validation.
stat, in_indices = validate_inputs(stat, in_indices, stat_path, in_indices_path)

print(f"Number of models: {stat.shape[0]}, Number of examples: {stat.shape[1]}, Augmentations: {stat.shape[2]}")
print(f"Stat shape: {stat.shape}, In indices shape: {in_indices.shape}")

Data loaded successfully.
Number of models: 32, Number of examples: 4096, Augmentations: 1
Stat shape: (32, 4096, 1), In indices shape: (32, 4096)


### Step 2: Specify a target model

Skip this part if you already have the target model statistics (`stat_target`) and shadow model statistics (`stat_in`, `stat_out`) separated.

If you already have these statistics, you can directly assign them as follows:

```python
# Uncomment and replace with your pre-separated statistics if available
# stat_target = ...  # Target model statistics
# stat_in = ...      # Shadow model statistics for in-shadow models
# stat_out = ...     # Shadow model statistics for out-shadow models
```
---

Alternatively, you can separate the target model statistics from the shadow model statistics by specifying the target model below.

We verify the fraction of NaN values to ensure the target statistics are available for the attack and the shadow model statistics are suitable for estimating the likelihood ratio.

In this example, stat_in.shape == stat_out.shape, but it is acceptable if stat_in.shape != stat_out.shape, provided the first dimension (the number of in or out shadow models) is not significantly different. For instance, a shape mismatch like (stat_in.shape[0], stat_out.shape[0]) = (31, 5) would lead to unreliable likelihood ratios.

In [3]:
# Uncomment and replace with your pre-separated statistics if available
# stat_target = ...  # Target model statistics
# stat_in = ...      # Shadow model statistics for in-shadow models
# stat_out = ...     # Shadow model statistics for out-shadow models

target_model_idx = 0 # Specify the target model index, here we consider the first model to be the target model

# Prepare statistics for LiRA attack by separating target and shadow model statistics.
stat_target, stat_in, stat_out = get_shadow_stats(stat, in_indices, target_model_idx)

# Check the fraction of NaN values in the separated statistics.
print("We expect that the shape of the stat_target misses the model dimension and the other two have one less model than before.")
print("We expect that the fraction of NaN values is 0 for the stat_target but around 0.5 for stat_in and stat_out.")
print(f"Shape of stat_target {stat_target.shape}, fraction of NaN {np.isnan(stat_target).mean():.6f}")
print(f"Shape of stat_in {stat_in.shape}, fraction of NaN {np.isnan(stat_in).mean():.6f}")
print(f"Shape of stat_out {stat_out.shape}, fraction of NaN {np.isnan(stat_out).mean():.6f}")

We expect that the shape of the stat_target misses the model dimension and the other two have one less model than before.
We expect that the fraction of NaN values is 0 for the stat_target but around 0.5 for stat_in and stat_out.
Shape of stat_target (4096, 1), fraction of NaN 0.000000
Shape of stat_in (31, 4096, 1), fraction of NaN 0.499929
Shape of stat_out (31, 4096, 1), fraction of NaN 0.500071


### Step 3: Compute LiRA Scores for the target model

The `run_simple_lira` function calculates the LiRA scores based on the separated `stat_target`, `stat_in`, and `stat_out`. 

Explanation of the options:
- **`fix_variance`**: If set to `True`, the variance of the estimated shadow model distribution is calculated globally, using all the examples. The recommended setting is `True` for $ M \leq 64 $ because it is more robust to the outliers that may be present in the case of using per-example variance with fewer shadow models.

- **`use_median`**: If set to `True`, the mean of the estimated shadow model distribution will be median instead of the mean. The recommended setting is `True` because it is more robust to outliers.

\* It is possible that other options enable more powerful attacks depending on the dataset and the training algorithms, so we recommend to test all the options.

In [4]:
# Compute LiRA scores using Gaussian distribution fitting.
scores = run_simple_lira(
    stat_target, stat_in, stat_out, fix_variance=True, use_median=True
)
print(f"The shape of the scores should be equivalent to the number of examples, it is {scores.shape}")

The shape of the scores should be equivalent to the number of examples, it is (4096,)


### Step 4: Check the scores

Here we look at the 10 highest and lowest scores, which correspond to the most confidently scored examples.

If the score is high, the likelihood ratio is high, therefore, the example is more likely to have been included in the training data.


In [5]:
sorted_scores = np.sort(scores)

print("10 highest scores:")
print(sorted_scores[-10:][::-1])

print("\n10 lowest scores:")
print(sorted_scores[:10])

10 highest scores:
[15.89181866  7.17269979  6.12227917  5.7344562   5.37757562  4.86912374
  4.82346579  4.5836783   4.50757554  4.1279392 ]

10 lowest scores:
[-21.66318825  -8.75017367  -8.42913916  -7.95758377  -7.22326303
  -5.8256999   -4.27887513  -4.27234507  -4.10178641  -3.8198234 ]
